# Solar Surface Convection — Implementation / 구현

**Paper**: Nordlund, Å., Stein, R. F., & Asplund, M. (2009). "Solar Surface Convection", *Living Rev. Solar Phys.*, 6, 2.

This notebook implements key physical concepts from the review:
이 노트북은 리뷰 논문의 핵심 물리 개념을 구현합니다:

1. **Solar stratification & scale heights / 태양 층화와 눈금높이** — Hydrostatic equilibrium, pressure/density profiles
2. **Granule scale selection / 과립 크기 선택** — Mass conservation constraint on convective cell size
3. **Convective energy flux balance / 대류 에너지 플럭스 균형** — Surface radiative loss vs. convective enthalpy flux
4. **Velocity spectrum / 속도 스펙트럼** — Multi-scale convection power spectrum
5. **p-mode excitation & frequency correction / p-mode 여기와 주파수 보정** — Turbulent pressure effect on resonant cavity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid
from scipy.constants import k as k_B, m_p, sigma, G

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

In [ ]:
# Solar constants / 태양 상수
M_sun = 1.989e30      # Solar mass [kg]
R_sun = 6.96e8         # Solar radius [m]
L_sun = 3.828e26       # Solar luminosity [W]
T_eff = 5778.0         # Effective temperature [K]
g_sun = G * M_sun / R_sun**2  # Surface gravity [m/s^2]

# Mean molecular weight (partially ionized hydrogen-helium mixture)
mu_neutral = 1.3       # Neutral H+He mixture
mu_ionized = 0.6       # Fully ionized H+He
chi_H = 13.6 * 1.602e-19  # Hydrogen ionization energy [J]

print(f"Solar surface gravity: g = {g_sun:.1f} m/s^2 = {g_sun*100:.1f} cm/s^2")
print(f"Effective temperature: T_eff = {T_eff:.0f} K")
print(f"Solar flux: F = sigma*T_eff^4 = {sigma * T_eff**4:.3e} W/m^2")

## Part 1: Solar Stratification & Pressure Scale Height / 태양 층화와 압력 눈금높이

From Eq. 7–9 of the paper, hydrostatic equilibrium gives:
논문의 Eq. 7–9로부터, 정역학적 평형에서:

$$-\frac{P}{\rho}\frac{\partial \ln P}{\partial z} = g_z \quad \Rightarrow \quad P = P_0 e^{-z/H_P}, \quad H_P = \frac{P}{\rho g_z} = \frac{k_B T}{\mu m_p g}$$

The scale height $H_P$ depends on temperature and mean molecular weight $\mu$. Near the solar surface, $T \approx 5800$ K and $\mu$ varies from ~1.3 (neutral) to ~0.6 (fully ionized), giving $H_P \approx 100$–300 km.

눈금높이 $H_P$는 온도와 평균 분자량 $\mu$에 의존합니다. 태양 표면 근처에서 $T \approx 5800$ K이고 $\mu$가 ~1.3(중성)에서 ~0.6(완전 이온화)까지 변하므로, $H_P \approx 100$–300 km입니다.

In [ ]:
def pressure_scale_height(T, mu, g=g_sun):
    """Pressure scale height H_P = k_B T / (mu * m_p * g).
    
    Args:
        T: Temperature [K]
        mu: Mean molecular weight [dimensionless]
        g: Surface gravity [m/s^2]
    
    Returns:
        H_P in meters
    """
    return k_B * T / (mu * m_p * g)


def build_stratification(z_range_m, T_profile, mu_profile, P0, g=g_sun):
    """Build hydrostatic stratification by integrating dP/dz = -rho*g.
    
    Args:
        z_range_m: Height array [m], positive upward from tau=1 surface
        T_profile: Temperature as function of z [K]
        mu_profile: Mean molecular weight as function of z
        P0: Pressure at z=0 [Pa]
        g: Surface gravity [m/s^2]
    
    Returns:
        P, rho arrays
    """
    dz = np.diff(z_range_m)
    P = np.zeros_like(z_range_m)
    rho = np.zeros_like(z_range_m)
    P[0] = P0
    rho[0] = P0 * mu_profile[0] * m_p / (k_B * T_profile[0])
    
    for i in range(len(dz)):
        # dP/dz = -rho * g  (z positive upward)
        rho_curr = P[i] * mu_profile[i] * m_p / (k_B * T_profile[i])
        rho[i] = rho_curr
        P[i+1] = P[i] - rho_curr * g * dz[i]
        if P[i+1] < 0:
            P[i+1] = P[i] * 0.01
    
    rho[-1] = P[-1] * mu_profile[-1] * m_p / (k_B * T_profile[-1])
    return P, rho


# Demonstrate scale height variation / 눈금높이 변화 시연
T_range = np.linspace(4000, 20000, 100)  # Temperature range [K]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scale height vs temperature for different mu
for mu, label in [(0.6, r'$\mu=0.6$ (fully ionized / 완전 이온화)'),
                   (1.0, r'$\mu=1.0$ (partially ionized / 부분 이온화)'),
                   (1.3, r'$\mu=1.3$ (neutral / 중성)')]:
    H = pressure_scale_height(T_range, mu) / 1e3  # Convert to km
    axes[0].plot(T_range, H, label=label, linewidth=2)

axes[0].set_xlabel('Temperature / 온도 [K]')
axes[0].set_ylabel('$H_P$ [km]')
axes[0].set_title('Pressure Scale Height vs Temperature\n압력 눈금높이 vs 온도')
axes[0].legend()
axes[0].axhline(y=150, color='gray', linestyle='--', alpha=0.5, label='~150 km (photosphere)')

# Build a simplified solar stratification
# Approximate temperature profile: photosphere to 20 Mm depth
z_km = np.linspace(-20000, 500, 2000)  # -20 Mm to +0.5 Mm
z_m = z_km * 1e3

# Simplified T profile (schematic)
T_profile = np.where(z_km < -2000,
                     5800 + (-z_km - 2000) * 0.4,  # Adiabatic interior
                     np.where(z_km < 0,
                              5800 + z_km * 0.3,   # Sub-photospheric
                              np.where(z_km < 300,
                                       4400 + (z_km / 300) * 0,  # T minimum
                                       4400 + (z_km - 300) * 3)))  # Chromosphere

# Simplified mu profile (transition from ionized to neutral)
mu_profile = np.where(z_km < -500, 0.6,
                      np.where(z_km < 100, 0.6 + (z_km + 500) / 600 * 0.7,
                               1.3))

P0 = 1.2e4  # Surface pressure ~120 dyn/cm^2 = 12000 Pa
P, rho = build_stratification(z_m, T_profile, mu_profile, P0)

# Plot density stratification (cf. Figure 1 of paper)
axes[1].semilogy(-z_km / 1000, rho, 'b-', linewidth=2)
axes[1].set_xlabel('Depth below surface / 표면 아래 깊이 [Mm]')
axes[1].set_ylabel(r'Density / 밀도 $\rho$ [kg/m$^3$]')
axes[1].set_title('Density Stratification (schematic)\n밀도 층화 (개요)')
axes[1].set_xlim(-0.5, 20)
axes[1].axvline(x=0, color='r', linestyle='--', alpha=0.7, label='Surface / 표면')
axes[1].legend()

plt.tight_layout()
plt.savefig('part1_stratification.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key values
H_surface = pressure_scale_height(5800, 1.0)
H_deep = pressure_scale_height(15000, 0.6)
print(f"\nScale height at surface (T=5800K, mu=1.0): H_P = {H_surface/1e3:.0f} km")
print(f"Scale height at 10 Mm depth (T=15000K, mu=0.6): H_P = {H_deep/1e3:.0f} km")
print(f"Density ratio over 20 Mm: ~{rho[0]/rho[-1]:.0e}")

## Part 2: Granule Scale Selection / 과립 크기 선택

From Eq. 36–38 of the paper, the granule size is determined by mass conservation:
논문의 Eq. 36–38로부터, 과립 크기는 질량 보존에 의해 결정됩니다:

$$\pi r^2 \rho v_z \approx 2\pi r H \rho v_H \quad \Rightarrow \quad r = 2H \frac{v_H}{v_z}$$

The minimum vertical velocity to carry the solar flux (Eq. 38):
태양 플럭스를 운반하기 위한 최소 수직 속도 (Eq. 38):

$$\sigma T_{\text{eff}}^4 \approx \rho V_z \left(\frac{5}{2}kT + x\chi\right)$$

where $x \approx 0.1$ is the hydrogen ionization fraction near the surface and $\chi = 13.6$ eV is the ionization energy.

In [ ]:
def granule_radius(H_p, v_h, v_z):
    """Granule radius from mass conservation (Eq. 37).
    
    Args:
        H_p: Pressure scale height [m]
        v_h: Horizontal velocity [m/s]
        v_z: Vertical velocity [m/s]
    
    Returns:
        Granule radius [m]
    """
    return 2 * H_p * (v_h / v_z)


def min_vertical_velocity(T, rho, x_ion=0.1):
    """Minimum vertical velocity to carry solar flux (Eq. 38).
    
    Args:
        T: Temperature [K]
        rho: Density [kg/m^3]
        x_ion: Hydrogen ionization fraction
    
    Returns:
        Minimum v_z [m/s]
    """
    F_solar = sigma * T_eff**4
    enthalpy = 2.5 * k_B * T + x_ion * chi_H
    return F_solar / (rho * enthalpy)


def sound_speed(T, mu=1.0, gamma=5/3):
    """Adiabatic sound speed.
    
    Args:
        T: Temperature [K]
        mu: Mean molecular weight
        gamma: Adiabatic index
    
    Returns:
        Sound speed [m/s]
    """
    return np.sqrt(gamma * k_B * T / (mu * m_p))


# Surface conditions / 표면 조건
T_surface = 5800  # K
rho_surface = 3e-4  # kg/m^3 (typical photospheric density)
mu_surface = 1.0

# Calculate key quantities
H_p = pressure_scale_height(T_surface, mu_surface)
v_z_min = min_vertical_velocity(T_surface, rho_surface)
c_s = sound_speed(T_surface, mu_surface)

print("=== Surface Conditions / 표면 조건 ===")
print(f"Scale height H_P = {H_p/1e3:.0f} km")
print(f"Minimum vertical velocity v_z = {v_z_min/1e3:.1f} km/s")
print(f"Sound speed c_s = {c_s/1e3:.1f} km/s")
print(f"Maximum horizontal velocity v_H ≈ c_s = {c_s/1e3:.1f} km/s")

# Granule radius for different velocity ratios
print("\n=== Granule Size Estimates / 과립 크기 추정 ===")
r_min = granule_radius(H_p, v_z_min, v_z_min)  # v_H = v_z
r_max = granule_radius(H_p, c_s, v_z_min)       # v_H = c_s (maximum)
r_typical = granule_radius(H_p, 4e3, v_z_min)    # v_H ~ 4 km/s (typical)

print(f"Minimum granule diameter (v_H = v_z): 2r = {2*r_min/1e3:.0f} km = {2*r_min/1e6:.2f} Mm")
print(f"Typical granule diameter (v_H ~ 4 km/s): 2r = {2*r_typical/1e3:.0f} km = {2*r_typical/1e6:.2f} Mm")
print(f"Maximum granule diameter (v_H = c_s): 2r = {2*r_max/1e3:.0f} km = {2*r_max/1e6:.2f} Mm")
print(f"\nObserved typical granule size: ~1 Mm (1000 km) ✓")

In [ ]:
# Convective cell size as a function of depth / 깊이에 따른 대류 셀 크기
# Deeper → larger H_P → larger cells (§3.3)

depths_km = np.array([0, 500, 1000, 2000, 5000, 10000, 20000])  # km
T_at_depth = np.array([5800, 8000, 12000, 25000, 60000, 100000, 200000])  # Approximate T [K]
mu_at_depth = np.array([1.0, 0.7, 0.6, 0.6, 0.6, 0.6, 0.6])

H_at_depth = pressure_scale_height(T_at_depth, mu_at_depth) / 1e3  # km

# Estimate cell size (r ~ few * H_P)
cell_diameter = 2 * 3 * H_at_depth / 1e3  # Mm (factor ~3 for typical v_H/v_z ratio)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scale height vs depth
axes[0].plot(depths_km / 1e3, H_at_depth, 'bo-', markersize=8, linewidth=2)
axes[0].set_xlabel('Depth / 깊이 [Mm]')
axes[0].set_ylabel('$H_P$ [km]')
axes[0].set_title('Scale Height vs Depth\n눈금높이 vs 깊이')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlim(0.1, 30)

# Cell size vs depth with observed scales marked
axes[1].plot(depths_km / 1e3, cell_diameter, 'rs-', markersize=8, linewidth=2,
             label='Estimated cell diameter / 추정 셀 직경')
axes[1].axhspan(0.5, 2.0, alpha=0.2, color='green', label='Granulation (~1 Mm)')
axes[1].axhspan(5, 10, alpha=0.2, color='orange', label='Mesogranulation (~5-10 Mm)')
axes[1].axhspan(20, 50, alpha=0.2, color='blue', label='Supergranulation (~30 Mm)')
axes[1].set_xlabel('Depth / 깊이 [Mm]')
axes[1].set_ylabel('Cell diameter / 셀 직경 [Mm]')
axes[1].set_title('Convective Cell Size vs Depth\n대류 셀 크기 vs 깊이')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlim(0.1, 30)
axes[1].legend(loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('part2_granule_scale.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 3: Multi-Scale Velocity Spectrum / 다중 스케일 속도 스펙트럼

The paper shows (§4.4, Figure 22) that the velocity spectrum $V(k) = \sqrt{kP(k)}$ is approximately linear in $k$ — velocity amplitudes decrease roughly inversely with wavenumber for scales larger than granulation. This means there is a smooth, continuous spectrum of motions rather than distinct "mesogranulation" or "supergranulation" modes.

논문은 (§4.4, Figure 22) 속도 스펙트럼 $V(k) = \sqrt{kP(k)}$가 $k$에 대해 거의 선형임을 보여줍니다 — 과립보다 큰 스케일에서 속도 진폭이 파수에 대략 반비례합니다. 이는 별개의 "mesogranulation"이나 "supergranulation" 모드가 아닌, 매끄러운 연속 스펙트럼을 의미합니다.

In [ ]:
def velocity_spectrum(k, V0, k0, alpha=1.0):
    """Model velocity spectrum V(k) = sqrt(k * P(k)).
    
    Following Figure 22: approximately linear in k for k > k_granulation,
    with a peak near granulation scale.
    V(k) ~ V0 * (k/k0) for k < k0 (rising part)
    V(k) ~ V0 * (k0/k)^alpha for k > k0 (declining part, alpha~1)
    
    Args:
        k: Spherical harmonic wavenumber
        V0: Peak velocity amplitude [m/s]
        k0: Peak wavenumber (granulation scale)
        alpha: Power law slope for large k
    
    Returns:
        V(k) in m/s
    """
    return V0 * (k / k0) / (1 + (k / k0)**(1 + alpha))


# Spherical harmonic wavenumber range
# l=1 corresponds to global scale (~700 Mm), l~1000 to granulation (~1 Mm)
ell = np.logspace(0, 4, 500)

# Conversion: physical scale L ~ 2*pi*R_sun / ell
L_Mm = 2 * np.pi * R_sun / (ell * 1e6)  # Physical scale [Mm]

# Model velocity spectrum
# Peak at granulation: l ~ 1000 (L ~ 4 Mm)
k0_gran = 1000
V0_peak = 3000  # ~3 km/s at granulation peak
V_spectrum = velocity_spectrum(ell, V0_peak, k0_gran)

fig, ax = plt.subplots(figsize=(10, 7))

# Plot velocity spectrum
ax.loglog(ell, V_spectrum, 'b-', linewidth=2.5, label='Model $V(k) = \\sqrt{kP(k)}$')

# Mark traditional scale labels
scales = {
    'Giant cells\n자이언트 셀': (10, 20),
    'Supergranulation\n초과립': (64, 200),
    'Mesogranulation\n중간과립': (300, 100),
    'Granulation\n과립': (1000, 3000),
}

colors = ['purple', 'blue', 'orange', 'green']
for (name, (ell_val, v_val)), color in zip(scales.items(), colors):
    ax.annotate(name, xy=(ell_val, v_val), fontsize=10, color=color,
                ha='center', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.15))

# Add observed data points (approximate from Figure 22)
ell_obs = np.array([10, 30, 64, 100, 300, 500, 1000, 2000, 5000])
v_obs = np.array([30, 80, 150, 250, 500, 800, 2500, 1500, 300])
ax.scatter(ell_obs, v_obs, c='red', s=60, zorder=5, marker='D',
           label='Approximate observed data\n관측 데이터 (근사)')

# Reference lines
ax.plot([1, 1000], [3, 3000], 'k--', alpha=0.3, linewidth=1.5)
ax.text(50, 80, '$V \\propto k$', fontsize=12, alpha=0.5, rotation=35)

ax.set_xlabel('Spherical Harmonic Degree $\\ell$', fontsize=13)
ax.set_ylabel('Velocity Amplitude $V(k) = \\sqrt{kP(k)}$ [m/s]', fontsize=13)
ax.set_title('Multi-Scale Velocity Spectrum (cf. Figure 22)\n다중 스케일 속도 스펙트럼', fontsize=14)
ax.set_xlim(1, 1e4)
ax.set_ylim(5, 1e4)
ax.legend(fontsize=11, loc='upper left')

# Add physical scale axis on top
ax2 = ax.twiny()
ell_ticks = [1, 10, 100, 1000, 10000]
ax2.set_xscale('log')
ax2.set_xlim(ax.get_xlim())
L_labels = [f"{2*np.pi*R_sun/(l*1e6):.0f}" for l in ell_ticks]
ax2.set_xticks(ell_ticks)
ax2.set_xticklabels(L_labels)
ax2.set_xlabel('Physical Scale / 물리적 크기 [Mm]', fontsize=13)

plt.tight_layout()
plt.savefig('part3_velocity_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key velocities at different scales
print("=== Velocity at Different Scales / 각 스케일의 속도 ===")
for name, ell_val in [('Giant cells (100 Mm)', 64),
                       ('Supergranulation (30 Mm)', 200),
                       ('Mesogranulation (5 Mm)', 1000),
                       ('Granulation (1 Mm)', 4000)]:
    v = velocity_spectrum(ell_val, V0_peak, k0_gran)
    print(f"  {name}: V ~ {v:.0f} m/s = {v/1e3:.1f} km/s")

## Part 4: p-mode Frequency Correction / p-mode 주파수 보정

The paper (§6.3) explains that convection modifies p-mode frequencies through two effects:
논문 (§6.3)은 대류가 두 가지 효과로 p-mode 주파수를 수정함을 설명합니다:

1. **Turbulent pressure** ($P_t \approx 0.15 P_g$) raises the atmosphere by ~$H_P/2$
   난류 압력이 대기를 ~$H_P/2$만큼 상승시킴

2. **Hiding of hot gas** (H⁻ opacity temperature sensitivity) raises it another ~$H_P/2$
   뜨거운 가스의 은닉(H⁻ 불투명도의 온도 의존성)이 추가로 ~$H_P/2$ 상승

Total effect: atmosphere extends by ~1 scale height → resonant cavity enlarges → p-mode frequencies decrease → better agreement with observations.
총 효과: 대기가 ~1 눈금높이 확장 → 공명 공동 증가 → p-mode 주파수 감소 → 관측과 더 나은 일치.

In [ ]:
def pmode_frequency_shift(nu_0, delta_R, R=R_sun):
    """Estimate p-mode frequency shift from cavity size change.
    
    For a resonant cavity, nu ~ c / (2L) where L is the cavity size.
    Fractional change: delta_nu/nu ~ -delta_R/R
    
    Args:
        nu_0: Original frequency [Hz]
        delta_R: Change in cavity size (atmosphere extension) [m]
        R: Solar radius [m]
    
    Returns:
        Frequency shift [Hz], shifted frequency [Hz]
    """
    delta_nu = -nu_0 * delta_R / R
    return delta_nu, nu_0 + delta_nu


# p-mode frequency range (typical: 1-5 mHz)
nu_mHz = np.linspace(1.0, 5.0, 100)
nu_Hz = nu_mHz * 1e-3

# Scale height at photosphere
H_P_surface = pressure_scale_height(5800, 1.0)  # ~170 km

# Two effects from §6.3:
# 1. Turbulent pressure: ~15% of gas pressure → ~H_P/2 extension
delta_R_turb = H_P_surface / 2
# 2. Hiding of hot gas: similar magnitude → ~H_P/2 extension
delta_R_hiding = H_P_surface / 2
# Total extension
delta_R_total = delta_R_turb + delta_R_hiding

print(f"=== p-mode Frequency Correction / p-mode 주파수 보정 ===")
print(f"Scale height at photosphere: H_P = {H_P_surface/1e3:.0f} km")
print(f"Extension from turbulent pressure: {delta_R_turb/1e3:.0f} km")
print(f"Extension from hot gas hiding: {delta_R_hiding/1e3:.0f} km")
print(f"Total atmosphere extension: {delta_R_total/1e3:.0f} km ≈ 1 H_P")
print(f"Fractional change: delta_R/R = {delta_R_total/R_sun:.2e}")

# Compute frequency shifts
delta_nu_1D, _ = pmode_frequency_shift(nu_Hz, 0)  # 1D model (no correction)
delta_nu_turb, _ = pmode_frequency_shift(nu_Hz, delta_R_turb)
delta_nu_total, nu_corrected = pmode_frequency_shift(nu_Hz, delta_R_total)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequency shift vs frequency
axes[0].plot(nu_mHz, delta_nu_turb * 1e6, 'b--', linewidth=2,
             label='Turbulent pressure only\n난류 압력만')
axes[0].plot(nu_mHz, delta_nu_total * 1e6, 'r-', linewidth=2,
             label='Total (turb. + hiding)\n합계 (난류 + 은닉)')
axes[0].set_xlabel('Frequency / 주파수 [mHz]')
axes[0].set_ylabel(r'Frequency shift $\delta\nu$ [$\mu$Hz]')
axes[0].set_title('p-mode Frequency Shift from Convection\n대류에 의한 p-mode 주파수 편이')
axes[0].legend()
axes[0].axhline(y=0, color='k', linewidth=0.5)

# Schematic of the effect on the resonant cavity
z_1D = np.linspace(0, 2.5, 100)  # Mm, depth below "surface"
z_3D = z_1D + delta_R_total / 1e6  # Shifted by ~H_P

P_1D = np.exp(-z_1D / (H_P_surface / 1e6))
P_3D = np.exp(-z_3D / (H_P_surface / 1e6))

axes[1].semilogy(z_1D, P_1D * 1e4, 'b-', linewidth=2, label='1D model / 1D 모델')
axes[1].semilogy(z_1D, P_3D * 1e4, 'r-', linewidth=2, label='3D model (shifted)\n3D 모델 (이동)')
axes[1].fill_between(z_1D, P_1D * 1e4, P_3D * 1e4, alpha=0.2, color='orange',
                      label=f'Extension ~{delta_R_total/1e3:.0f} km\n확장')
axes[1].set_xlabel('Depth / 깊이 [Mm]')
axes[1].set_ylabel('Pressure (arb. units) / 압력 (임의 단위)')
axes[1].set_title('Atmosphere Extension by Convection\n대류에 의한 대기 확장 (cf. Fig. 42)')
axes[1].legend(fontsize=10)
axes[1].set_xlim(0, 2.5)

plt.tight_layout()
plt.savefig('part4_pmode_correction.png', dpi=150, bbox_inches='tight')
plt.show()

# Print shifts for key frequencies
print("\n=== Frequency Shifts / 주파수 편이 ===")
for nu in [2.0, 3.0, 4.0, 5.0]:
    delta, _ = pmode_frequency_shift(nu * 1e-3, delta_R_total)
    print(f"  nu = {nu:.1f} mHz: delta_nu = {delta*1e6:.1f} μHz")

## Part 5: Solar Abundance Comparison / 태양 풍부도 비교

The most impactful result of this review: the 3D model-based solar C, N, O abundances are significantly lower than previous 1D-based values (§5.3, Table 1). This section visualizes the abundance revision and the resulting "solar abundance crisis."

이 리뷰의 가장 영향력 있는 결과: 3D 모델 기반 태양 C, N, O 풍부도가 이전 1D 기반 값보다 현저히 낮습니다 (§5.3, Table 1). 이 섹션은 풍부도 수정과 그로 인한 "태양 풍부도 위기"를 시각화합니다.

In [ ]:
# Solar abundance data from Table 1 and §5.3 / 태양 풍부도 데이터

# log epsilon values (log N_X/N_H + 12)
elements = ['C', 'N', 'O', 'Fe']

# Different determinations
abundances = {
    'Anders & Grevesse 1989\n(이전 표준)': {
        'C': 8.56, 'N': 8.05, 'O': 8.93, 'Fe': 7.67
    },
    'Grevesse & Sauval 1998': {
        'C': 8.52, 'N': 7.92, 'O': 8.83, 'Fe': 7.50
    },
    '3D (This review)\n3D (이 리뷰)': {
        'C': 8.39, 'N': 7.80, 'O': 8.66, 'Fe': 7.45
    },
}

errors_3D = {'C': 0.05, 'N': 0.05, 'O': 0.05, 'Fe': 0.05}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart comparison
x = np.arange(len(elements))
width = 0.25
colors = ['#2196F3', '#FF9800', '#E91E63']

for i, (label, vals) in enumerate(abundances.items()):
    values = [vals[e] for e in elements]
    bars = axes[0].bar(x + i*width, values, width, label=label, color=colors[i], alpha=0.8)
    
    # Add error bars for 3D
    if '3D' in label:
        errs = [errors_3D[e] for e in elements]
        axes[0].errorbar(x + i*width, values, yerr=errs, fmt='none', color='black',
                        capsize=4, linewidth=1.5)

axes[0].set_xlabel('Element / 원소', fontsize=13)
axes[0].set_ylabel(r'$\log \epsilon$ (= $\log N_X/N_H + 12$)', fontsize=13)
axes[0].set_title('Solar Abundance Revision\n태양 풍부도 수정', fontsize=14)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(elements, fontsize=13)
axes[0].legend(fontsize=9, loc='upper right')
axes[0].set_ylim(7.0, 9.2)

# Difference plot (3D - AG89)
diffs = {e: abundances['3D (This review)\n3D (이 리뷰)'][e] - abundances['Anders & Grevesse 1989\n(이전 표준)'][e]
         for e in elements}

colors_diff = ['red' if d < 0 else 'blue' for d in diffs.values()]
bars = axes[1].bar(elements, list(diffs.values()), color=colors_diff, alpha=0.7, edgecolor='black')

# Add value labels
for bar, val in zip(bars, diffs.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.02,
                f'{val:.2f} dex', ha='center', va='top', fontweight='bold', fontsize=12)

axes[1].set_xlabel('Element / 원소', fontsize=13)
axes[1].set_ylabel(r'$\Delta \log \epsilon$ (3D $-$ AG89)', fontsize=13)
axes[1].set_title('Abundance Change (3D vs AG89)\n풍부도 변화 (3D vs AG89)', fontsize=14)
axes[1].axhline(y=0, color='k', linewidth=1)
axes[1].set_ylim(-0.35, 0.05)

plt.tight_layout()
plt.savefig('part5_abundances.png', dpi=150, bbox_inches='tight')
plt.show()

# Metallicity comparison
Z_AG89 = 0.0194
Z_GS98 = 0.0170
Z_3D = 0.0122

print("=== Metallicity Comparison / 금속량 비교 ===")
print(f"Anders & Grevesse (1989): Z = {Z_AG89}")
print(f"Grevesse & Sauval (1998): Z = {Z_GS98}")
print(f"3D (Asplund et al. 2005): Z = {Z_3D}")
print(f"Reduction factor: {Z_3D/Z_AG89:.1%} of AG89 value ({(1-Z_3D/Z_AG89):.0%} decrease)")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Pressure scale height / 압력 눈금높이 | $H_P = k_B T / (\mu m_p g) \approx 150$ km at surface | Same fundamental relation; used in all stellar atmosphere codes / 동일한 기본 관계; 모든 항성 대기 코드에서 사용 |
| Granule size selection / 과립 크기 선택 | $r = 2H(v_H/v_z) \approx 1$ Mm | Confirmed by modern high-res observations (SST, DKIST) / 현대 고해상도 관측(SST, DKIST)으로 확인 |
| Velocity spectrum / 속도 스펙트럼 | $V(k) = \sqrt{kP(k)}$ — smooth, no distinct scales above granulation | SDO/HMI observations confirm continuous spectrum / SDO/HMI 관측이 연속 스펙트럼 확인 |
| Solar abundances / 태양 풍부도 | $\log\epsilon_C=8.39$, $\log\epsilon_N=7.80$, $\log\epsilon_O=8.66$ | Asplund et al. (2021): slightly revised upward but still lower than AG89 / 소폭 상향이나 여전히 AG89보다 낮음 |
| p-mode correction / p-mode 보정 | ~1 $H_P$ atmosphere extension reduces frequency discrepancy | Standard treatment in modern helioseismology / 현대 helioseismology의 표준 처리 |
| 3D model atmospheres / 3D 모델 대기 | Stagger, CO5BOLD codes | MURaM, Bifrost, Mancha3D — extended to chromosphere/corona / 채층/코로나까지 확장 |